# TD4 — Système simplifié d'évaluation des foyers (RSU)

## Présentation
Ce notebook reprend le système de collecte, de sauvegarde CSV, de calcul d'un score et de génération d'un rapport d'éligibilité développé dans le TP.

**Attention :** les coefficients et le seuil ci-dessous sont conservés tels qu'ils figurent dans le travail fourni. Ce notebook est un exercice pédagogique ; il ne constitue pas un outil officiel de décision d'éligibilité au Registre Social Unifié.

## 1. Constantes et imports

In [ ]:
import csv
from dataclasses import dataclass
from pathlib import Path

Kzg_urban = 0.194
Kzg_rural = 0.328
Kmu = 9.825
Kmr = 8.695

C1_urban, C1_rural = 0.182, 0.190
C2_urban, C2_rural = 0.151, 0.193
C3_urban, C3_rural = -0.177, -0.138
C4_urban, C4_rural = 0.00011, 0.000145
C5_urban, C5_rural = 0.0475, 0.129

ELIGIBILITY_THRESHOLD = 9.32

CSV_HEADER = [
    "application_id", "region", "Statut_professionnel",
    "Couverture_santé", "Taille_du_ménage",
    "dépenses_eau_electricite_par_personne", "Type_de_logement"
]

## 2. Modèle de données et validation

In [ ]:
@dataclass
class Household:
    application_id: str
    region: str
    Statut_professionnel: float
    Couverture_santé: float
    Taille_du_ménage: float
    dépenses_eau_electricite_par_personne: float
    Type_de_logement: int


def validate_household(h: Household) -> None:
    if not h.application_id.strip():
        raise ValueError("L'identifiant du foyer est obligatoire.")
    if h.region not in {"Urban", "Rural"}:
        raise ValueError("La région doit être 'Urban' ou 'Rural'.")
    if not 0 <= h.Statut_professionnel <= 1:
        raise ValueError("Statut_professionnel doit être compris entre 0 et 1.")
    if not 0 <= h.Couverture_santé <= 1:
        raise ValueError("Couverture_santé doit être comprise entre 0 et 1.")
    if h.Taille_du_ménage < 1:
        raise ValueError("La taille du ménage doit être au moins égale à 1.")
    if h.dépenses_eau_electricite_par_personne < 0:
        raise ValueError("Les dépenses ne peuvent pas être négatives.")
    if h.Type_de_logement < 0:
        raise ValueError("Le nombre de chambres ne peut pas être négatif.")

## 3. Calcul du score (formule conservée du TP)

In [ ]:
def compute_household_score(h: Household) -> float:
    validate_household(h)

    if h.region == "Urban":
        base = Kzg_urban + Kmu
        c1, c2, c3, c4, c5 = C1_urban, C2_urban, C3_urban, C4_urban, C5_urban
    else:
        base = Kzg_rural + Kmr
        c1, c2, c3, c4, c5 = C1_rural, C2_rural, C3_rural, C4_rural, C5_rural

    score = (
        base
        + c1 * h.Statut_professionnel
        + c2 * h.Couverture_santé
        + c3 * h.Taille_du_ménage
        + c4 * h.dépenses_eau_electricite_par_personne
        + c5 * h.Type_de_logement
    )
    return round(score, 4)

## 4. Sauvegarde et lecture du fichier CSV

In [ ]:
def save_to_csv(filepath: str | Path, household: Household) -> None:
    path = Path(filepath)
    file_exists = path.exists()

    with path.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=CSV_HEADER)
        if not file_exists:
            writer.writeheader()
        writer.writerow(household.__dict__)


def ingest_applications(filepath: str | Path) -> list[Household]:
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    households = []
    with path.open("r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            household = Household(
                application_id=row["application_id"],
                region=row["region"],
                Statut_professionnel=float(row["Statut_professionnel"]),
                Couverture_santé=float(row["Couverture_santé"]),
                Taille_du_ménage=float(row["Taille_du_ménage"]),
                dépenses_eau_electricite_par_personne=float(
                    row["dépenses_eau_electricite_par_personne"]
                ),
                Type_de_logement=int(float(row["Type_de_logement"]))
            )
            validate_household(household)
            households.append(household)
    return households

## 5. Rapport d'évaluation

In [ ]:
def generate_eligibility_report(households: list[Household]) -> None:
    if not households:
        print("Aucun enregistrement à afficher.")
        return

    print("=" * 72)
    print("RAPPORT PÉDAGOGIQUE — ÉVALUATION DES FOYERS".center(72))
    print(f"Seuil utilisé dans le TP : score < {ELIGIBILITY_THRESHOLD}".center(72))
    print("=" * 72)
    print(f"{'ID':<14} {'Région':<10} {'Score':<10} {'Résultat'}")
    print("-" * 72)

    eligible = 0
    for household in households:
        score = compute_household_score(household)
        status = "Éligible (selon le TP)" if score < ELIGIBILITY_THRESHOLD else "Non éligible (selon le TP)"
        eligible += score < ELIGIBILITY_THRESHOLD
        print(f"{household.application_id:<14} {household.region:<10} {score:<10.4f} {status}")

    print("-" * 72)
    print(f"Nombre de foyers : {len(households)}")
    print(f"Éligibles selon le seuil du TP : {eligible}/{len(households)} ({eligible/len(households)*100:.1f}%)")

## 6. Exemple reproductible et tests

In [ ]:
sample_households = [
    Household("TEST01", "Urban", 0.2, 0.0, 5.0, 1000.0, 2),
    Household("TEST02", "Rural", 0.6, 1.0, 4.0, 500.0, 3),
]

for household in sample_households:
    validate_household(household)
    print(household.application_id, compute_household_score(household))

# Test de validation
try:
    invalid = Household("BAD01", "Unknown", 0.2, 0.0, 2.0, 100.0, 2)
    compute_household_score(invalid)
except ValueError:
    print("Test de validation réussi.")
else:
    raise AssertionError("Une région invalide aurait dû être refusée.")

generate_eligibility_report(sample_households)

## 7. Utilisation facultative du CSV

La cellule suivante montre comment sauvegarder puis relire les exemples. Elle crée un fichier `population_data.csv` dans le répertoire courant du notebook.

In [ ]:
csv_path = Path("population_data.csv")

# Décommenter pour créer le fichier de démonstration :
# for household in sample_households:
#     save_to_csv(csv_path, household)
#
# households_from_csv = ingest_applications(csv_path)
# generate_eligibility_report(households_from_csv)

## 8. Collecte interactive (optionnelle)

La collecte par `input()` est conservée comme piste d'extension, mais elle n'est pas exécutée automatiquement dans ce notebook afin de permettre une exécution complète et reproductible sur GitHub/Colab.

**Compétences mobilisées :** constantes, dataclasses, validation, fonctions, CSV, exceptions, calcul d'un score et génération d'un rapport.